# Tea-leaf disease segmentation with SegFormer

Cleaned from the original Colab experiment. Four classes: background, algal leaf spot, brown blight, grey blight.

**Before running:** install requirements and set `ROOT` to a dataset containing `train/`, `valid/`, `test/` folders with image/mask pairs. Outputs are cleared; results from the project report have not been revalidated here.


## Setup and configuration


In [ ]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

ROOT = Path("data/leaf_segmentation")  # Set this to your local dataset directory

TRAIN_DIR = ROOT / "train"
VAL_DIR   = ROOT / "valid"
TEST_DIR  = ROOT / "test"

print("Train exists:", TRAIN_DIR.exists())
print("Valid exists:", VAL_DIR.exists())
print("Test exists:", TEST_DIR.exists())

SEED = 42
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 25
LR = 2e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

CLASS_NAMES = {
    0: "background",
    1: "algal leaf spot",
    2: "brown blight",
    3: "grey blight"
}

NUM_CLASSES = 4

CLASS_COLORS = {
    0: (0, 0, 0),
    1: (255, 0, 0),
    2: (0, 255, 0),
    3: (0, 0, 255),
}

id2label = CLASS_NAMES
label2id = {v: k for k, v in id2label.items()}

def build_pairs_from_split(split_dir: Path, split_name: str):
    files = list(split_dir.iterdir())

    image_files = []
    for f in files:
        if not f.is_file():
            continue
        if f.name == "_classes.csv":
            continue
        if f.name.endswith("_mask.png"):
            continue
        if f.suffix.lower() in IMAGE_EXTS:
            image_files.append(f)

    records = []
    for img_path in sorted(image_files):
        mask_name = f"{img_path.stem}_mask.png"
        mask_path = split_dir / mask_name

        if mask_path.exists():
            records.append({
                "split": split_name,
                "image_name": img_path.name,
                "mask_name": mask_name,
                "image_path": str(img_path),
                "mask_path": str(mask_path),
            })

    return pd.DataFrame(records)



## Data discovery and mask encoding


In [ ]:
train_df = build_pairs_from_split(TRAIN_DIR, "train")
val_df   = build_pairs_from_split(VAL_DIR, "valid")
test_df  = build_pairs_from_split(TEST_DIR, "test")

print("Train pairs:", len(train_df))
print("Valid pairs:", len(val_df))
print("Test pairs:", len(test_df))

df_all = pd.concat([train_df, val_df, test_df], ignore_index=True)
display(df_all.head())

def inspect_mask(mask_path):
    arr = np.array(Image.open(mask_path))
    print("Mask:", mask_path)
    print("shape:", arr.shape, "dtype:", arr.dtype)

    if arr.ndim == 2:
        print("unique values:", np.unique(arr)[:50])
    else:
        reshaped = arr.reshape(-1, arr.shape[-1])
        uniq = np.unique(reshaped, axis=0)
        print("unique RGB rows (first 20):")
        print(uniq[:20])

sample_paths = df_all["mask_path"].sample(min(4, len(df_all)), random_state=SEED).tolist()
for p in sample_paths:
    inspect_mask(p)
    print("-" * 60)

def raw_mask_to_class_indices(mask_pil):
    arr = np.array(mask_pil)

    if arr.ndim == 2:
        class_mask = arr.astype(np.int64)
    else:
        signal = arr[..., :3].max(axis=2)
        class_mask = signal.astype(np.int64)

    unique_vals = np.unique(class_mask)
    valid_vals = {0, 1, 2, 3}
    if not set(unique_vals).issubset(valid_vals):
        raise ValueError(f"Unexpected mask values found: {unique_vals}. Expected only 0,1,2,3.")

    return class_mask

def decode_mask_to_color(mask_np):
    h, w = mask_np.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, rgb in CLASS_COLORS.items():
        color[mask_np == cls] = rgb
    return color

sample_df = df_all.sample(min(4, len(df_all)), random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(len(sample_df), 3, figsize=(14, 4 * len(sample_df)))
if len(sample_df) == 1:
    axes = np.array([axes])

for i, row in sample_df.iterrows():
    img = np.array(Image.open(row["image_path"]).convert("RGB"))
    mask_raw = Image.open(row["mask_path"])
    class_mask = raw_mask_to_class_indices(mask_raw)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(row["image_name"])
    axes[i, 0].axis("off")

    axes[i, 1].imshow(class_mask, vmin=0, vmax=3, cmap="viridis")
    axes[i, 1].set_title("Class mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(decode_mask_to_color(class_mask))
    axes[i, 2].set_title("Colorized mask")
    axes[i, 2].axis("off")

    print(row["mask_name"], "unique classes:", np.unique(class_mask))

plt.tight_layout()
plt.show()

processor = SegformerImageProcessor(
    do_resize=False,
    do_reduce_labels=False
)



## Dataset and loaders


In [ ]:
class TeaDiseaseSegFormerDataset(Dataset):
    def __init__(self, df, img_size=512, augment=False, processor=None):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.augment = augment
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def _resize(self, image, mask):
        image = image.resize((self.img_size, self.img_size), Image.BILINEAR)
        mask  = mask.resize((self.img_size, self.img_size), Image.NEAREST)
        return image, mask

    def _augment(self, image, mask):
        if random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask  = mask.transpose(Image.FLIP_LEFT_RIGHT)

        if random.random() < 0.3:
            image = image.transpose(Image.FLIP_TOP_BOTTOM)
            mask  = mask.transpose(Image.FLIP_TOP_BOTTOM)

        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            image = image.rotate(angle, resample=Image.BILINEAR)
            mask  = mask.rotate(angle, resample=Image.NEAREST)

        image = ImageEnhance.Brightness(image).enhance(random.uniform(0.9, 1.1))
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.9, 1.1))
        image = ImageEnhance.Color(image).enhance(random.uniform(0.9, 1.1))

        return image, mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        mask  = Image.open(row["mask_path"])

        image, mask = self._resize(image, mask)

        if self.augment:
            image, mask = self._augment(image, mask)

        mask_np = raw_mask_to_class_indices(mask).astype(np.uint8)

        encoded = self.processor(
            images=image,
            segmentation_maps=mask_np,
            return_tensors="pt"
        )

        pixel_values = encoded["pixel_values"].squeeze(0)
        labels = encoded["labels"].squeeze(0).long()

        return {
            "pixel_values": pixel_values,
            "labels": labels,
            "image_name": row["image_name"],
            "mask_name": row["mask_name"],
            "split": row["split"],
        }

train_ds = TeaDiseaseSegFormerDataset(train_df, img_size=IMG_SIZE, augment=True, processor=processor)
val_ds   = TeaDiseaseSegFormerDataset(val_df,   img_size=IMG_SIZE, augment=False, processor=processor)
test_ds  = TeaDiseaseSegFormerDataset(test_df,  img_size=IMG_SIZE, augment=False, processor=processor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

batch = next(iter(train_loader))
print(batch["pixel_values"].shape, batch["labels"].shape)

sample = next(iter(train_loader))
img = sample["pixel_values"][0].numpy().transpose(1, 2, 0)

mean = np.array(processor.image_mean).reshape(1, 1, 3)
std = np.array(processor.image_std).reshape(1, 1, 3)
img_vis = np.clip(img * std + mean, 0, 1)

lbl = sample["labels"][0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_vis)
axes[0].set_title("Image")
axes[0].axis("off")

axes[1].imshow(lbl, vmin=0, vmax=3, cmap="viridis")
axes[1].set_title("Multiclass label")
axes[1].axis("off")

plt.tight_layout()
plt.show()



## Model and metrics


In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(DEVICE)

print("SegFormer ready.")

def compute_multiclass_metrics_from_logits(logits, labels, num_classes=4, eps=1e-6):
    logits_up = F.interpolate(
        logits,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    preds = torch.argmax(logits_up, dim=1)

    dice_per_class = {}
    iou_per_class = {}

    dice_vals = []
    iou_vals = []

    for cls in range(1, num_classes):
        pred_c = (preds == cls).float()
        targ_c = (labels == cls).float()

        tp = (pred_c * targ_c).sum()
        fp = (pred_c * (1 - targ_c)).sum()
        fn = ((1 - pred_c) * targ_c).sum()

        dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
        iou  = (tp + eps) / (tp + fp + fn + eps)

        dice_per_class[f"dice_class_{cls}"] = float(dice.cpu())
        iou_per_class[f"iou_class_{cls}"] = float(iou.cpu())

        dice_vals.append(float(dice.cpu()))
        iou_vals.append(float(iou.cpu()))

    pixel_acc = float((preds == labels).float().mean().cpu())

    out = {
        "pixel_acc": pixel_acc,
        "mean_dice": float(np.mean(dice_vals)),
        "mean_iou": float(np.mean(iou_vals)),
    }
    out.update(dice_per_class)
    out.update(iou_per_class)
    return out

def run_epoch_segformer(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_count = 0
    metric_sums = None

    for batch in tqdm(loader, leave=False):
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            if is_train:
                loss.backward()
                optimizer.step()

        bs = pixel_values.size(0)
        total_loss += loss.item() * bs
        total_count += bs

        metrics = compute_multiclass_metrics_from_logits(outputs.logits.detach(), labels.detach(), num_classes=NUM_CLASSES)

        if metric_sums is None:
            metric_sums = {k: 0.0 for k in metrics.keys()}

        for k in metric_sums:
            metric_sums[k] += metrics[k] * bs

    results = {"loss": total_loss / total_count}
    for k in metric_sums:
        results[k] = metric_sums[k] / total_count

    return results

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

history = []
best_val_dice = -1.0


## Training


In [ ]:
best_model_path = "outputs/best_segformer_multiclass_leaf.pt"
os.makedirs("outputs", exist_ok=True)

patience = 6
bad_epochs = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_res = run_epoch_segformer(model, train_loader, optimizer=optimizer)
    val_res   = run_epoch_segformer(model, val_loader, optimizer=None)

    scheduler.step(val_res["mean_dice"])

    history.append({
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        **{f"train_{k}": v for k, v in train_res.items()},
        **{f"val_{k}": v for k, v in val_res.items()},
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_res['loss']:.4f} | "
        f"val_loss={val_res['loss']:.4f} | "
        f"val_mean_dice={val_res['mean_dice']:.4f} | "
        f"val_mean_iou={val_res['mean_iou']:.4f}"
    )

    if val_res["mean_dice"] > best_val_dice:
        best_val_dice = val_res["mean_dice"]
        torch.save(model.state_dict(), best_model_path)
        bad_epochs = 0
        print("Saved best SegFormer model.")
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping triggered.")
        break

history_df = pd.DataFrame(history)
display(history_df.head())

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_mean_dice"], marker="o", label="train mean dice")
plt.plot(history_df["epoch"], history_df["val_mean_dice"], marker="o", label="val mean dice")
plt.xlabel("Epoch")
plt.ylabel("Mean Dice")
plt.title("SegFormer Training History")
plt.grid(True)
plt.legend()
plt.show()

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(DEVICE)



## Test evaluation


In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

print("Loaded best model from:", best_model_path)

def evaluate_test_set(model, loader):
    rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing"):
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            names = batch["image_name"]

            outputs = model(pixel_values=pixel_values)

            for i in range(pixel_values.size(0)):
                one_logits = outputs.logits[i:i+1]
                one_label = labels[i:i+1]
                m = compute_multiclass_metrics_from_logits(one_logits, one_label, num_classes=NUM_CLASSES)

                row = {"image_name": names[i]}
                row.update(m)
                rows.append(row)

    return pd.DataFrame(rows)

test_metrics_df = evaluate_test_set(model, test_loader)
test_metrics_df.to_csv("outputs/test_metrics_segformer_multiclass.csv", index=False)

print("Test metrics mean:")
print(test_metrics_df.drop(columns=["image_name"]).mean())

def predict_batch(model, pixel_values):
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        logits = F.interpolate(
            outputs.logits,
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False
        )
        preds = torch.argmax(logits, dim=1)
    return preds



## Prediction exports


In [ ]:
def make_overlay_multiclass(image_np, pred_mask_np, alpha=0.35):
    overlay = image_np.copy()
    color_mask = decode_mask_to_color(pred_mask_np).astype(np.float32) / 255.0

    lesion_region = pred_mask_np > 0
    overlay[lesion_region] = (1 - alpha) * overlay[lesion_region] + alpha * color_mask[lesion_region]
    return np.clip(overlay, 0, 1)

SAVE_DIR = Path("outputs/prediction_outputs_segformer_multiclass")
MASK_DIR = SAVE_DIR / "masks"
COLOR_MASK_DIR = SAVE_DIR / "color_masks"
OVERLAY_DIR = SAVE_DIR / "overlays"

MASK_DIR.mkdir(parents=True, exist_ok=True)
COLOR_MASK_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Saving predictions"):
        pixel_values = batch["pixel_values"].to(DEVICE)
        names  = batch["image_name"]

        preds = predict_batch(model, pixel_values)

        for i in range(pixel_values.size(0)):
            image_name = names[i]
            stem = Path(image_name).stem

            img = pixel_values[i].cpu().numpy().transpose(1, 2, 0)
            mean = np.array(processor.image_mean).reshape(1, 1, 3)
            std = np.array(processor.image_std).reshape(1, 1, 3)
            img_np = np.clip(img * std + mean, 0, 1)

            pred_np = preds[i].cpu().numpy().astype(np.uint8)

            mask_path = MASK_DIR / f"{stem}_pred.png"
            color_mask_path = COLOR_MASK_DIR / f"{stem}_pred_color.png"
            overlay_path = OVERLAY_DIR / f"{stem}_overlay.png"

            Image.fromarray(pred_np).save(mask_path)
            Image.fromarray(decode_mask_to_color(pred_np)).save(color_mask_path)

            overlay = make_overlay_multiclass(img_np, pred_np)
            Image.fromarray((overlay * 255).astype(np.uint8)).save(overlay_path)

print("Saved masks to:", MASK_DIR)
print("Saved color masks to:", COLOR_MASK_DIR)
print("Saved overlays to:", OVERLAY_DIR)

overlay_files = sorted(list(OVERLAY_DIR.glob("*.png")))
print("Total overlays:", len(overlay_files))

n_show = min(4, len(overlay_files))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
if n_show == 1:
    axes = [axes]

for ax, f in zip(axes, overlay_files[:n_show]):
    img = np.array(Image.open(f).convert("RGB"))
    ax.imshow(img)
    ax.set_title(f.name)
    ax.axis("off")

plt.tight_layout()


## Image-level interpretation and visualization


In [ ]:
plt.show()

def get_image_level_disease_from_mask(pred_mask):
    vals, counts = np.unique(pred_mask, return_counts=True)
    area_map = dict(zip(vals.tolist(), counts.tolist()))

    disease_areas = {cls: area_map.get(cls, 0) for cls in [1, 2, 3]}
    total_disease_pixels = sum(disease_areas.values())

    if total_disease_pixels == 0:
        return {
            "predicted_disease": "no disease detected",
            "dominant_class": 0,
            "algal_leaf_spot_pixels": 0,
            "brown_blight_pixels": 0,
            "grey_blight_pixels": 0,
            "total_disease_pixels": 0
        }

    dominant_class = max(disease_areas, key=disease_areas.get)

    return {
        "predicted_disease": CLASS_NAMES[dominant_class],
        "dominant_class": dominant_class,
        "algal_leaf_spot_pixels": disease_areas[1],
        "brown_blight_pixels": disease_areas[2],
        "grey_blight_pixels": disease_areas[3],
        "total_disease_pixels": total_disease_pixels
    }

def predict_one(model, image_path, img_size=512):
    image = Image.open(image_path).convert("RGB")
    image_r = image.resize((img_size, img_size), Image.BILINEAR)

    mask_dummy = np.zeros((img_size, img_size), dtype=np.uint8)
    encoded = processor(images=image_r, segmentation_maps=mask_dummy, return_tensors="pt")
    x = encoded["pixel_values"].to(DEVICE)

    with torch.no_grad():
        outputs = model(pixel_values=x)
        logits = F.interpolate(
            outputs.logits,
            size=(img_size, img_size),
            mode="bilinear",
            align_corners=False
        )
        pred = torch.argmax(logits, dim=1)[0].cpu().numpy().astype(np.uint8)

    return np.asarray(image_r), pred

rows = []

for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc="Scoring all images"):
    img_np, pred_np = predict_one(model, row["image_path"], img_size=IMG_SIZE)
    disease_info = get_image_level_disease_from_mask(pred_np)

    rows.append({
        "split": row["split"],
        "image_name": row["image_name"],
        **disease_info
    })

image_scores_df = pd.DataFrame(rows)
image_scores_df.to_csv("/content/image_level_disease_predictions_segformer.csv", index=False)
display(image_scores_df.head(20))

print(image_scores_df["predicted_disease"].value_counts())

sample_rows = image_scores_df.sample(min(5, len(image_scores_df)), random_state=SEED)

fig, axes = plt.subplots(len(sample_rows), 3, figsize=(12, 4 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = np.array([axes])

for i, (_, row) in enumerate(sample_rows.iterrows()):
    image_path = str(ROOT / row["split"] / row["image_name"])
    img_np, pred_np = predict_one(model, image_path, img_size=IMG_SIZE)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(row["image_name"])
    axes[i, 0].axis("off")

    axes[i, 1].imshow(decode_mask_to_color(pred_np))
    axes[i, 1].set_title("Predicted class mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(make_overlay_multiclass(img_np / 255.0, pred_np))
    axes[i, 2].set_title(row["predicted_disease"])
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()